# Doc2Vec với Pretrained Model (Sentence Transformers)

Notebook này được chuyển từ nội dung trong `BAO_CAO.md` và `run_doc2vec.py`.
Mục tiêu: tạo document embeddings và so sánh hiệu năng phân loại giữa Sentence Transformers và Gensim Doc2Vec.

## 1) Cài đặt và import thư viện

In [1]:
# Nếu thiếu package, bỏ comment và chạy cell dưới:
# !pip install -q sentence-transformers scikit-learn gensim numpy

import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.datasets import fetch_20newsgroups
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

## 2) Cấu hình thí nghiệm

In [2]:
DATA_HOME = './data'
os.makedirs(DATA_HOME, exist_ok=True)

CATEGORIES = ['sci.med', 'sci.space', 'rec.sport.baseball', 'talk.politics.misc']
MAX_CHARS = 3000

print('=' * 60)
print('DOC2VEC VOI PRETRAINED MODEL (SENTENCE TRANSFORMERS)')
print('=' * 60)

DOC2VEC VOI PRETRAINED MODEL (SENTENCE TRANSFORMERS)


## 3) Load dữ liệu 20 Newsgroups

In [3]:
train_data = fetch_20newsgroups(
    subset='train',
    categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
    random_state=42,
    data_home=DATA_HOME,
)

test_data = fetch_20newsgroups(
    subset='test',
    categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
    random_state=42,
    data_home=DATA_HOME,
)

train_texts = [doc[:MAX_CHARS] for doc in train_data.data]
test_texts = [doc[:MAX_CHARS] for doc in test_data.data]
train_labels = train_data.target
test_labels = test_data.target

print(f'Train: {len(train_texts)} samples')
print(f'Test: {len(test_texts)} samples')
print(f'Categories: {CATEGORIES}')

Train: 2249 samples
Test: 1497 samples
Categories: ['sci.med', 'sci.space', 'rec.sport.baseball', 'talk.politics.misc']


## 4) Sentence Transformers + Logistic Regression

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model: all-MiniLM-L6-v2 (384D)')

train_vectors = model.encode(train_texts, batch_size=32, show_progress_bar=False)
test_vectors = model.encode(test_texts, batch_size=32, show_progress_bar=False)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(train_vectors, train_labels)
acc_st = accuracy_score(test_labels, clf.predict(test_vectors))

print(f'Encoded train shape: {train_vectors.shape}')
print(f'Encoded test shape: {test_vectors.shape}')
print(f'Sentence Transformers Accuracy: {acc_st * 100:.2f}%')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8854.18it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: all-MiniLM-L6-v2 (384D)
Encoded train shape: (2249, 384)
Encoded test shape: (1497, 384)
Sentence Transformers Accuracy: 87.71%


## 5) Gensim Doc2Vec (PV-DM và PV-DBOW)

In [5]:
tagged_docs = [TaggedDocument(words=t.split(), tags=[str(i)]) for i, t in enumerate(train_texts)]

# PV-DM
doc2vec_dm = Doc2Vec(vector_size=100, window=5, min_count=2, workers=4, dm=1, epochs=20)
doc2vec_dm.build_vocab(tagged_docs)
doc2vec_dm.train(tagged_docs, total_examples=doc2vec_dm.corpus_count, epochs=doc2vec_dm.epochs)

# PV-DBOW
doc2vec_dbow = Doc2Vec(vector_size=100, window=5, min_count=2, workers=4, dm=0, epochs=20)
doc2vec_dbow.build_vocab(tagged_docs)
doc2vec_dbow.train(tagged_docs, total_examples=doc2vec_dbow.corpus_count, epochs=doc2vec_dbow.epochs)

# Tạo vector train/test
train_vecs_dm = np.array([doc2vec_dm.dv[str(i)] for i in range(len(train_texts))])
train_vecs_dbow = np.array([doc2vec_dbow.dv[str(i)] for i in range(len(train_texts))])
test_vecs_dm = np.array([doc2vec_dm.infer_vector(t.split()) for t in test_texts])
test_vecs_dbow = np.array([doc2vec_dbow.infer_vector(t.split()) for t in test_texts])

# Đánh giá PV-DM
clf_dm = LogisticRegression(max_iter=1000, random_state=42)
clf_dm.fit(train_vecs_dm, train_labels)
acc_dm = accuracy_score(test_labels, clf_dm.predict(test_vecs_dm))

# Đánh giá PV-DBOW
clf_dbow = LogisticRegression(max_iter=1000, random_state=42)
clf_dbow.fit(train_vecs_dbow, train_labels)
acc_dbow = accuracy_score(test_labels, clf_dbow.predict(test_vecs_dbow))

print(f'PV-DM Accuracy: {acc_dm * 100:.2f}%')
print(f'PV-DBOW Accuracy: {acc_dbow * 100:.2f}%')

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


PV-DM Accuracy: 65.33%
PV-DBOW Accuracy: 74.55%


## 6) Tổng hợp kết quả

In [6]:
print('=' * 60)
print('KET QUA SO SANH')
print('=' * 60)
print(f'| Model                  | Accuracy |')
print(f'|-----------------------|----------|')
print(f'| Sentence Transformers | {acc_st * 100:>6.2f}% |')
print(f'| Gensim PV-DBOW        | {acc_dbow * 100:>6.2f}% |')
print(f'| Gensim PV-DM          | {acc_dm * 100:>6.2f}% |')
print('=' * 60)

print(f'\nCai thien ST so voi PV-DM: {(acc_st - acc_dm) * 100:+.2f}%')
print(f'Cai thien ST so voi PV-DBOW: {(acc_st - acc_dbow) * 100:+.2f}%')

KET QUA SO SANH
| Model                  | Accuracy |
|-----------------------|----------|
| Sentence Transformers |  87.71% |
| Gensim PV-DBOW        |  74.55% |
| Gensim PV-DM          |  65.33% |

Cai thien ST so voi PV-DM: +22.38%
Cai thien ST so voi PV-DBOW: +13.16%


## 7) Kết luận ngắn

- Sentence Transformers thường cho chất lượng cao hơn nhờ pretrained transformer embeddings.
- Gensim Doc2Vec vẫn là baseline nhẹ, dễ triển khai.
- Với dữ liệu tiếng Anh, `all-MiniLM-L6-v2` là lựa chọn hiệu quả giữa tốc độ và độ chính xác.